Import data from the bronze tables

In [0]:
from pyspark.sql import functions as F
CATALOG = "global_electronics_retailer_dataset"




In [0]:
# List all Bronze table in global_electronics_retailer_dataset Catalog

spark.sql("SHOW TABLES IN `global_electronics_retailer_dataset`.bronze").show()

In [0]:
# Load all Bronze table in global_electronics_retailer_dataset Catalog
sales = spark.read.table(f"{CATALOG}.bronze.sales")
customers = spark.read.table(f"{CATALOG}.bronze.customers")
products = spark.read.table(f"{CATALOG}.bronze.products")
stores = spark.read.table(f"{CATALOG}.bronze.stores")
exchange_rates = spark.read.table(f"{CATALOG}.bronze.exchange_rates")


In [0]:
sales.dtypes

Common functions

In [0]:
# Clean & Trim string dtype columns

def clean_text_columns(df):
    for name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(name, F.trim(F.col(name)))
            df = df.withColumn(name,F.when(F.col(name)=="",None).otherwise(F.col(name)))
    return df


# Cast to integer 

def cast_to_integer(df, columns):
    for c in columns:
        df = df.withColumn(c,F.col(c).cast("int"))
    return df


# cast to date

def cast_to_date(df, columns):
    for c in columns:
        df = df.withColumn(c,F.to_date(F.col(c),'M/d/yyyy'))
    return df

def cast_to_decimal(df, columns):
    for c in columns:
        df = df.withColumn(c,F.col(c).cast("decimal(10,3)"))
    return df


Sales Silver Transformation

In [0]:
# Cleaning Sales Table

sales.printSchema()
display(sales.limit(5))


In [0]:

columns_to_integer = ['quantity']
sales = sales.withColumn('ingestion_date', F.to_date(F.col("_ingest_time")))
columns_to_date = ['delivery_date','order_date','ingestion_date']

sales = clean_text_columns(sales)
sales = cast_to_integer(sales,columns_to_integer)
sales = cast_to_date(sales,columns_to_date)

sales = sales.drop("_sourcefile","_ingest_time")




In [0]:
sales.printSchema()

In [0]:
# Save silver layer table - sales

sales.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.sales")

Customer Silver Transformation

In [0]:
customers.printSchema()

In [0]:
display(customers.limit(5))

In [0]:
customers = customers.withColumn('ingestion_date', F.to_date(F.col("_ingest_time")))
columns_to_date = ['ingestion_date', 'birthday']

customers = clean_text_columns(customers)
customers = cast_to_date(customers,columns_to_date)
customers = customers.drop("_sourcefile","_ingest_time")

In [0]:
customers.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.customers")


Exchange Rates Silver Transformation

In [0]:
exchange_rates.printSchema()

In [0]:
display(exchange_rates.limit(5))

In [0]:
exchange_rates = exchange_rates.withColumn('ingestion_date', F.to_date(F.col("_ingest_time")))
date_columns = ["date","ingestion_date"]
exchange_rates = clean_text_columns(exchange_rates)
exchange_rates = cast_to_date(exchange_rates,date_columns)
exchange_rates = cast_to_decimal(exchange_rates,["exchange"])
exchange_rates = exchange_rates.drop("_sourcefile","_ingest_time")


In [0]:
exchange_rates.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.exchange_rates")

Products - Silver Transformation

In [0]:
products.printSchema()
display(products)

In [0]:



products = products.withColumn('ingestion_date', F.to_date(F.col("_ingest_time")))
date_columns = ["ingestion_date"]
products = products.withColumn("unit_cost_usd", F.regexp_replace(F.col("unit_cost_usd"), r"[\$,]", ""))
products = products.withColumn("unit_price_usd", F.regexp_replace(F.col("unit_cost_usd"), r"[\$,]", ""))
products = clean_text_columns(products)
products = cast_to_date(products,date_columns)
products = cast_to_decimal(products,["unit_cost_usd","unit_price_usd"])
products = products.drop("_sourcefile","_ingest_time")



In [0]:
products.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.products")

Stores - Silver Transformation

In [0]:
stores.printSchema()
display(stores)

In [0]:
stores = stores.withColumn('ingestion_date', F.to_date(F.col("_ingest_time")))
date_columns = ["ingestion_date", "open_date"]

stores = cast_to_integer(stores,["square_meters"])
stores = clean_text_columns(stores)
stores = cast_to_date(stores,date_columns)
stores = stores.drop("_sourcefile","_ingest_time")


In [0]:
stores.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.stores")
